# BDS HiveServer2 — Kerberos JDBC read

Read a Hive table from Oracle Big Data Service (BDS) over JDBC with Kerberos auth. Uses Java's built-in `Krb5LoginModule` (no `kinit` binary needed).

**Network prerequisites** (the skill SKILL.md spells these out in detail):
1. AIDP cluster's hidden VCN must be peered with the BDS subnet, with TCP open on the HS2 port.
2. Cross-VCN DNS so the BDS hostname resolves from the cluster (or use IP addresses).
3. The KDC hosts in your `krb5.conf` must be reachable from the cluster on UDP/TCP 88.

If you're running this from a workbench whose cluster CANNOT reach the BDS subnet, this notebook will fail at the connectivity probe (cell 2). Run from a peered workbench, or set up VCN peering first.

## 1. Configuration

Set these env vars (or hard-code below). Defaults match the example BDS cluster shipped with this skill — replace with your own.

In [ ]:
import os

os.environ.setdefault('BDS_HS2_HOST',      'nitishun0-0.rgroverprdpub1.rgroverprd.oraclevcn.com')
os.environ.setdefault('BDS_HS2_PORT',      '10010')
os.environ.setdefault('BDS_DATABASE',      'test')
os.environ.setdefault('BDS_TABLE',         'test_sample')
os.environ.setdefault('BDS_HS2_PRINCIPAL', 'hive/nitishun0-0.rgroverprdpub1.rgroverprd.oraclevcn.com@NITISH.ORACLE.COM')
os.environ.setdefault('BDS_KEYTAB_PATH',   '/tmp/hive.service.keytab')
# Optional — point at a /tmp/krb5.conf you uploaded if /etc/krb5.conf doesn't list your KDCs.
# os.environ.setdefault('BDS_KRB5_CONF', '/tmp/krb5.conf')

for k in ['BDS_HS2_HOST','BDS_HS2_PORT','BDS_DATABASE','BDS_TABLE','BDS_HS2_PRINCIPAL','BDS_KEYTAB_PATH']:
    print(f'{k} = {os.environ[k]}')

## 2. Pre-flight — TCP probe to HS2

Fail fast if the network isn't there. If this prints `TCP FAIL`, stop and fix the VCN peering / DNS / NSG rules before continuing.

In [ ]:
import socket
host = os.environ['BDS_HS2_HOST']
port = int(os.environ['BDS_HS2_PORT'])
try:
    s = socket.create_connection((host, port), timeout=5)
    print(f'TCP OK to {host}:{port}')
    s.close()
except Exception as e:
    raise SystemExit(f'TCP FAIL to {host}:{port} — fix VCN peering / DNS first ({e})')

## 3. Runtime-load the Hive JDBC driver

Cluster has no `org.apache.hive.jdbc.HiveDriver` pre-installed; runtime-load the standalone JAR from Maven Central.

In [ ]:
from oracle_ai_data_platform_connectors.jdbc import runtime_load_hive_driver

runtime_load_hive_driver(
    spark,
    jar_path='/tmp/hive-jdbc-standalone.jar',
)

## 4. Kerberos login via JAAS keytab

Pure-Java equivalent of `kinit -kt <keytab> <principal>`. The TGT lives in the JVM Subject and the Hive JDBC driver picks it up automatically when it sees `auth=kerberos` in the URL.

In [ ]:
from oracle_ai_data_platform_connectors.jdbc import kerberos_login_via_jaas

kerberos_login_via_jaas(
    spark,
    keytab_path=os.environ['BDS_KEYTAB_PATH'],
    principal=os.environ['BDS_HS2_PRINCIPAL'],
    krb5_conf_path=os.environ.get('BDS_KRB5_CONF'),
    # debug=True,    # uncomment for KDC trace
)
print('Kerberos login OK')

## 5. Read the Hive table

In [ ]:
from oracle_ai_data_platform_connectors.jdbc import build_hive_jdbc_url

url = build_hive_jdbc_url(
    hs2_host=os.environ['BDS_HS2_HOST'],
    hs2_port=int(os.environ['BDS_HS2_PORT']),
    database=os.environ['BDS_DATABASE'],
    hs2_principal=os.environ['BDS_HS2_PRINCIPAL'],
)
print('JDBC URL:', url)

df = (spark.read.format('jdbc')
      .option('url',       url)
      .option('driver',    'org.apache.hive.jdbc.HiveDriver')
      .option('dbtable',   os.environ['BDS_TABLE'])
      .option('fetchsize', '10000')
      .load())
df.printSchema()
df.show(10)
row_count = df.count()
print('rows:', row_count)

## 6. Result summary

In [ ]:
import json, time
summary = {
    'connector': 'aidp-bds-hive',
    'auth':      'Kerberos via JAAS keytab login',
    'host':      os.environ['BDS_HS2_HOST'],
    'port':      int(os.environ['BDS_HS2_PORT']),
    'database':  os.environ['BDS_DATABASE'],
    'table':     os.environ['BDS_TABLE'],
    'rows':      row_count,
    'schema':    [(f.name, str(f.dataType)) for f in df.schema.fields],
    'timestamp_utc': int(time.time()),
}
print(json.dumps(summary, indent=2))